## Resume Skill Extraction

This notebook applies the finalized skill extraction and classification pipeline
to all resumes in the processed resume dataset.

In [3]:
import pandas as pd
import re

# Load cleaned resume dataset
resume_df = pd.read_csv("../data/processed/resumes_clean.csv")

print("Number of resumes:", len(resume_df))
print("Columns:")
print(resume_df.columns.tolist())

Number of resumes: 2484
Columns:
['ID', 'Resume_str', 'Resume_html', 'Category', 'text_length', 'html_length', 'clean_resume', 'raw_length', 'clean_length', 'retention_ratio']


In [4]:
# Load the cleaned skill vocabulary
skills_df = pd.read_csv("../data/processed/skills_clean.csv")

# Create a set for fast skill lookup
skill_set = set(skills_df["clean_skill"].dropna().str.lower())

print("Number of skills in vocabulary:", len(skill_set))

Number of skills in vocabulary: 36167


In [5]:
# Remove numeric-only skills
skills_extraction = skills_df[
    ~skills_df["clean_skill"].str.fullmatch(r"\d+")
].copy()


# Generic words that should not normally be treated as skills
generic_words = {
    "advanced", "analysis", "analytical", "codes", "collateral",
    "conflict", "customer", "data", "design", "features",
    "general", "government", "health", "hospital", "hotel",
    "insurance", "interpret", "layout", "management", "office",
    "operations", "paperwork", "participation", "plan", "policies",
    "prevention", "print", "public", "ratings", "records",
    "reporting", "reviews", "safety", "sales", "service",
    "skills", "statistics", "system", "teams", "terminology",
    "training", "transactions", "website", "organization",
    "professional", "responsibilities", "experience", "work",
    "working", "software"
}


# Important short/technical skills that should always be preserved
protected_skills = {
    "python", "java", "javascript", "c", "c++", "c#",
    "f#", "g++", "sql", "html", "css", "php",
    "ruby", "swift", "kotlin", "go", "r", "matlab",
    "scala", "perl", "bash", "powershell", "git",
    "github", "gitlab", "npm", "node.js", "react.js",
    "vue", "angular", "mongodb", "mysql", "postgresql",
    "oracle", "redis", "numpy", "pandas", "tensorflow",
    "pytorch", "keras", "tableau", "power bi", "docker",
    "kubernetes", "aws", "azure", "gcp", "linux",
    "android", "ios", "sap", "excel", "jira",
    "front-end"
}


def is_valid_extraction_skill(skill):
    skill = skill.lower().strip()

    # Keep multi-word skills
    if len(skill.split()) > 1:
        return True

    # Keep known technical/important short skills
    if skill in protected_skills:
        return True

    # Remove generic single words
    if skill in generic_words:
        return False

    return True


# Apply final extraction filtering
skills_extraction = skills_extraction[
    skills_extraction["clean_skill"].apply(is_valid_extraction_skill)
].reset_index(drop=True)


# Create final lookup set
skill_set = set(skills_extraction["clean_skill"])

print("Final extraction vocabulary:", len(skill_set))

Final extraction vocabulary: 36105


## Skill Extraction Functions
These Functions are tested in 03_Skill_extraction_File

In [18]:

def normalize_skill(skill):
    skill = skill.lower().strip()

    if skill in skill_aliases:
        return skill_aliases[skill]

    return skill


def normalize_text_aliases(text, skill_aliases):
    text = str(text).lower()

    aliases_sorted = sorted(
        skill_aliases.keys(),
        key=len,
        reverse=True
    )

    pattern = re.compile(
        r'(?<![a-z0-9])(' +
        '|'.join(re.escape(a) for a in aliases_sorted) +
        r')(?![a-z0-9])'
    )

    return pattern.sub(
        lambda m: skill_aliases[m.group(0)],
        text
    )


def extract_skills(text, skill_set, max_words=5):
    text = str(text).lower()

    # Keep characters important for technical skills
    text = re.sub(
        r'[^a-z0-9+#.\- ]+',
        ' ',
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [w.strip('.') for w in words]
    words = [w for w in words if w]

    found_skills = set()

    # Search longer phrases first
    for n in range(max_words, 0, -1):

        for i in range(len(words) - n + 1):

            phrase = " ".join(words[i:i+n])

            if phrase in skill_set:
                found_skills.add(phrase)

    return sorted(found_skills)


def remove_overlapping_skills(skills):

    skills = sorted(
        skills,
        key=lambda x: len(x.split()),
        reverse=True
    )

    final_skills = []

    for skill in skills:

        skill_words = skill.split()
        is_subskill = False

        for existing_skill in final_skills:

            existing_words = existing_skill.split()
            n = len(skill_words)

            for i in range(
                len(existing_words) - n + 1
            ):

                if existing_words[i:i+n] == skill_words:
                    is_subskill = True
                    break

            if is_subskill:
                break

        if not is_subskill:
            final_skills.append(skill)

    return sorted(final_skills)

## Skill Alias Normalization


In [19]:


skill_aliases = {

    # Frontend
    "react": "react.js",
    "reactjs": "react.js",
    "react js": "react.js",

    "vuejs": "vue",
    "vue js": "vue",

    "angularjs": "angular",
    "angular js": "angular",

    # Backend
    "nodejs": "node.js",
    "node js": "node.js",

    # Machine Learning / AI
    "scikit learn": "scikit-learn",
    "scikit learn library": "scikit-learn",

    "tensorflow library": "tensorflow",
    "pytorch library": "pytorch",

    "ml": "machine learning",
    "ai": "artificial intelligence",

    # Data
    "data analytics": "data analysis",

    # Databases
    "mongo": "mongodb",
    "mongo db": "mongodb",
    "postgre sql": "postgresql",

    # Programming
    "c sharp": "c#",
    "cpp": "c++",

    # Cloud
    "amazon web services": "aws",
    "microsoft azure": "azure",
    "google cloud platform": "gcp",

    # Version control
    "git hub": "github",

    # Power BI
    "powerbi": "power bi"
}

## Skill Classification References

In [22]:

technical_skills_reference = {
    "python", "java", "javascript", "c", "c++", "c#",
    "f#", "php", "ruby", "swift", "kotlin", "go", "r",
    "matlab", "scala", "perl", "bash", "powershell",

    "html", "css", "react.js", "vue", "angular",
    "node.js", "front-end",

    "sql", "mysql", "postgresql", "mongodb", "oracle", "redis",

    "numpy", "pandas", "scikit-learn", "tensorflow",
    "pytorch", "keras",

    "machine learning", "deep learning",
    "artificial intelligence", "data analysis",
    "data analytics", "natural language processing",
    "computer vision",

    "git", "github", "gitlab",
    "docker", "kubernetes",

    "aws", "azure", "gcp",
    "linux", "android", "ios",

    "tableau", "power bi", "excel"
}


software_tools_reference = {
    "facebook",
    "holidex",
    "micros",
    "opera reservation system",
    "opera",
    "icd-9"
}


professional_domain_reference = {
    "administration",
    "advertising",
    "analytical skills",
    "assessment",
    "budgeting",
    "certification",
    "commerce",
    "compensation",
    "computer skills",
    "conflict resolution",
    "customer loyalty",
    "customer satisfaction",
    "customer service management",
    "daily operations",
    "dental insurance",
    "documentation",
    "employee relations",
    "executive support",
    "financial management",
    "front office",
    "government relations",
    "healthcare",
    "hospitality",
    "human resources",
    "inventory control",
    "labor relations",
    "leadership",
    "leave of absence",
    "loss prevention",
    "management accounting",
    "marketing",
    "medical billing",
    "medical terminology",
    "morale",
    "payroll",
    "pension",
    "performance reviews",
    "presentations",
    "productivity",
    "programming",
    "public relations",
    "purchasing",
    "service delivery",
    "skilled multi-tasker",
    "supervisor",
    "team management",
    "time management",
    "trade shows"
}


def classify_skill(skill):

    if skill in technical_skills_reference:
        return "technical"

    if skill in software_tools_reference:
        return "software/tool"

    if skill in professional_domain_reference:
        return "professional/domain"

    return "other"

In [9]:
# ==========================================
# Complete Skill Extraction + Classification
# ==========================================

def extract_and_classify_skills(text, skill_set, max_words=5):

    # 1. Normalize aliases in resume text
    normalized_text = normalize_text_aliases(
        text,
        skill_aliases
    )

    # 2. Extract skills
    extracted = extract_skills(
        normalized_text,
        skill_set,
        max_words=max_words
    )

    # 3. Normalize extracted skill names
    normalized = [
        normalize_skill(skill)
        for skill in extracted
    ]

    # 4. Remove duplicates
    normalized = list(set(normalized))

    # 5. Remove overlapping skills
    final_skills = remove_overlapping_skills(
        normalized
    )

    # 6. Classify skills
    classified = {
        "technical": [],
        "software/tool": [],
        "professional/domain": [],
        "other": []
    }

    for skill in final_skills:

        category = classify_skill(skill)

        classified[category].append(skill)

    return final_skills, classified

In [10]:
# ==========================================
# Test on One Real Resume
# ==========================================

real_resume = resume_df.iloc[0]["clean_resume"]

final_skills, classified_skills = extract_and_classify_skills(
    real_resume,
    skill_set
)

print("Total skills:", len(final_skills))

print("\nTechnical Skills:")
for skill in classified_skills["technical"]:
    print("-", skill)

print("\nSoftware / Tools:")
for skill in classified_skills["software/tool"]:
    print("-", skill)

print("\nProfessional / Domain:")
for skill in classified_skills["professional/domain"]:
    print("-", skill)

print("\nOther:")
for skill in classified_skills["other"]:
    print("-", skill)

Total skills: 56

Technical Skills:
- data analysis
- front-end
- swift

Software / Tools:
- facebook
- holidex
- icd-9
- micros
- opera reservation system

Professional / Domain:
- administration
- advertising
- analytical skills
- assessment
- budgeting
- certification
- commerce
- compensation
- computer skills
- conflict resolution
- customer loyalty
- customer satisfaction
- customer service management
- daily operations
- dental insurance
- documentation
- employee relations
- executive support
- financial management
- front office
- government relations
- healthcare
- hospitality
- human resources
- inventory control
- labor relations
- leadership
- leave of absence
- loss prevention
- management accounting
- marketing
- medical billing
- medical terminology
- morale
- payroll
- pension
- performance reviews
- presentations
- productivity
- programming
- public relations
- purchasing
- service delivery
- skilled multi-tasker
- supervisor
- team management
- time management
- tra

In [11]:
# ==========================================
# Test Skill Extraction on 10 Resumes
# ==========================================

test_results = []

for index, row in resume_df.head(10).iterrows():

    final_skills, classified = extract_and_classify_skills(
        row["clean_resume"],
        skill_set
    )

    test_results.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "total_skills": len(final_skills),
        "technical_skills": classified["technical"],
        "software_tools": classified["software/tool"],
        "professional_domain": classified["professional/domain"],
        "other": classified["other"]
    })


test_df = pd.DataFrame(test_results)

print("Tested resumes:", len(test_df))
print("\nSkill counts:")
print(test_df[["ID", "Category", "total_skills"]])

Tested resumes: 10

Skill counts:
         ID Category  total_skills
0  16852973       HR            56
1  22323967       HR            76
2  33176873       HR            85
3  27018550       HR            36
4  17812897       HR            92
5  11592605       HR            70
6  25824789       HR            68
7  15375009       HR            72
8  11847784       HR            67
9  32896934       HR            63


In [12]:
# Display extracted skills for the first test resume

first_test = test_df.iloc[0]

print("Resume ID:", first_test["ID"])
print("Category:", first_test["Category"])
print("Total skills:", first_test["total_skills"])

print("\nTechnical:")
print(first_test["technical_skills"])

print("\nSoftware / Tools:")
print(first_test["software_tools"])

print("\nProfessional / Domain:")
print(first_test["professional_domain"])

print("\nOther:")
print(first_test["other"])

Resume ID: 16852973
Category: HR
Total skills: 56

Technical:
['data analysis', 'front-end', 'swift']

Software / Tools:
['facebook', 'holidex', 'icd-9', 'micros', 'opera reservation system']

Professional / Domain:
['administration', 'advertising', 'analytical skills', 'assessment', 'budgeting', 'certification', 'commerce', 'compensation', 'computer skills', 'conflict resolution', 'customer loyalty', 'customer satisfaction', 'customer service management', 'daily operations', 'dental insurance', 'documentation', 'employee relations', 'executive support', 'financial management', 'front office', 'government relations', 'healthcare', 'hospitality', 'human resources', 'inventory control', 'labor relations', 'leadership', 'leave of absence', 'loss prevention', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'morale', 'payroll', 'pension', 'performance reviews', 'presentations', 'productivity', 'programming', 'public relations', 'purchasing', 'service delivery

In [13]:
# ==========================================
# Extract Skills from All Resumes
# ==========================================

all_results = []

for index, row in resume_df.iterrows():

    final_skills, classified = extract_and_classify_skills(
        row["clean_resume"],
        skill_set
    )

    all_results.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "total_skills": len(final_skills),
        "technical_skills": ", ".join(classified["technical"]),
        "software_tools": ", ".join(classified["software/tool"]),
        "professional_domain": ", ".join(classified["professional/domain"]),
        "other": ", ".join(classified["other"])
    })

    # Progress indicator
    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1} / {len(resume_df)} resumes")


resume_skills_df = pd.DataFrame(all_results)

print("\nExtraction completed!")
print("Total resumes processed:", len(resume_skills_df))

Processed 100 / 2484 resumes
Processed 200 / 2484 resumes
Processed 300 / 2484 resumes
Processed 400 / 2484 resumes
Processed 500 / 2484 resumes
Processed 600 / 2484 resumes
Processed 700 / 2484 resumes
Processed 800 / 2484 resumes
Processed 900 / 2484 resumes
Processed 1000 / 2484 resumes
Processed 1100 / 2484 resumes
Processed 1200 / 2484 resumes
Processed 1300 / 2484 resumes
Processed 1400 / 2484 resumes
Processed 1500 / 2484 resumes
Processed 1600 / 2484 resumes
Processed 1700 / 2484 resumes
Processed 1800 / 2484 resumes
Processed 1900 / 2484 resumes
Processed 2000 / 2484 resumes
Processed 2100 / 2484 resumes
Processed 2200 / 2484 resumes
Processed 2300 / 2484 resumes
Processed 2400 / 2484 resumes

Extraction completed!
Total resumes processed: 2484


In [14]:
# ==========================================
# Inspect Final Resume Skills Dataset
# ==========================================

print("Shape:", resume_skills_df.shape)

print("\nColumns:")
print(resume_skills_df.columns.tolist())

print("\nFirst 5 rows:")
display(resume_skills_df.head())

Shape: (2484, 7)

Columns:
['ID', 'Category', 'total_skills', 'technical_skills', 'software_tools', 'professional_domain', 'other']

First 5 rows:


,ID,Category,total_skills,technical_skills,software_tools,professional_domain,other
0,16852973,HR,56,"data analysis, front-end, swift","facebook, holidex, icd-9, micros, opera reserv...","administration, advertising, analytical skills...",
1,22323967,HR,76,,,"advertising, human resources, presentations, p...","adobe photoshop, affinity, articles, backgroun..."
2,33176873,HR,85,excel,,"administration, advertising, commerce, front o...","access, agents, audit, automation, budget mana..."
3,27018550,HR,36,,,"administration, conflict resolution, customer ...","access, call control, call monitoring, call qu..."
4,17812897,HR,92,excel,,"conflict resolution, employee relations, human...","access, benefits administration, business unit..."


In [15]:
# ==========================================
# Basic Validation
# ==========================================

print("Total resumes:", len(resume_skills_df))

print("Missing IDs:", resume_skills_df["ID"].isna().sum())

print("Missing categories:", resume_skills_df["Category"].isna().sum())

print("Average skills per resume:",
      resume_skills_df["total_skills"].mean())

print("Minimum skills:",
      resume_skills_df["total_skills"].min())

print("Maximum skills:",
      resume_skills_df["total_skills"].max())

Total resumes: 2484
Missing IDs: 0
Missing categories: 0
Average skills per resume: 68.06280193236715
Minimum skills: 0
Maximum skills: 226


In [16]:
# ==========================================
# Save Resume Skill Extraction Results
# ==========================================

output_path = "../data/processed/resume_skills.csv"

resume_skills_df.to_csv(
    output_path,
    index=False
)

print("Saved successfully!")
print("File:", output_path)

Saved successfully!
File: ../data/processed/resume_skills.csv


In [17]:
# Verify saved file

import os

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", round(os.path.getsize(output_path) / 1024, 2), "KB")

File exists: True
File size: 2203.98 KB
